```markdown
SMS Spam Classification Project

This project implements and compares multiple Natural Language Processing (NLP) techniques to classify SMS messages as either **Ham** (legitimate) or Spam.

# Dataset Description
We use the SMS Spam Collection Dataset containing 5,572 English SMS messages.
v1 (label): Class of the message (`ham` or `spam`).
v2 (message): The raw text content of the SMS.

# Feature Extraction Methods
Count Vectorizer (Bag of Words): Binary word-occurrence representation.
TF-IDF Vectorizer: Term Frequency-Inverse Document Frequency capturing word importance.
Average Word2Vec: Semantics-preserving dense vector representations built with Gensim.

# ML
Multinomial Naive Bayes (MultinomialNB): Fast probabilistic classifier (used with Bag of Words & TF-IDF).
Random Forest Classifier: Robust ensemble classifier (used with average Word2Vec embedding features).
```

In [1]:
import pandas as pd
import numpy as np

import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "spam.csv"

messages = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS,
                            "uciml/sms-spam-collection-dataset",
                            file_path,
                            pandas_kwargs={"encoding" : "latin1"}
                            )

messages.head()

Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [2]:
messages.shape

(5572, 5)

In [3]:
messages = messages.drop(columns={"Unnamed: 2","Unnamed: 3","Unnamed: 4"})

In [4]:
messages.rename(columns={"v1":"label","v2":"message"},inplace=True)

In [5]:
messages.drop_duplicates()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [83]:
print(messages['label'].value_counts())

label
ham     4825
spam     747
Name: count, dtype: int64


In [6]:
messages.isnull().sum()

,0
label,0
message,0


In [7]:
import re
import nltk

from nltk.corpus import stopwords

nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [8]:
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [9]:
corpus = []

for i in range(0,len(messages)):
  review  = re.sub('[^a-zA-Z]', ' ', messages['message'][i])
  review = review.lower()
  review = review.split()

  review  = [ps.stem(word) for word in review if not word in stopwords.words('english')]
  review = ' '.join(review)
  corpus.append(review)

Count Vectorizer (word -> vector) (binary=true)

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=2500,binary=True,ngram_range=(1,2))
X = cv.fit_transform(corpus).toarray()

In [11]:
y = pd.get_dummies(messages['label'])
y = y.iloc[:,1].values

In [12]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train , y_test = train_test_split(X, y, random_state=42, test_size=0.2)

In [13]:
from sklearn.naive_bayes import MultinomialNB
spam_detect_model = MultinomialNB().fit(X_train, y_train)

In [14]:
y_pred = spam_detect_model.predict(X_test)

In [15]:
score = spam_detect_model.score(X_test,y_test)
print(score)

0.9802690582959641


In [16]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

       False       0.98      0.99      0.99       965
        True       0.96      0.89      0.92       150

    accuracy                           0.98      1115
   macro avg       0.97      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115



TF IDF VECTORIZER (Text -> Vector)

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
tv = TfidfVectorizer(max_features=2500,ngram_range=(1,2))
X = tv.fit_transform(corpus).toarray()

In [18]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
spam_classification = MultinomialNB().fit(X_train,y_train)
y_pred = spam_classification.predict(X_test)
score = spam_classification.score(X_test,y_test)
print(score)

0.97847533632287


In [19]:
print(classification_report(y_pred,y_test))

              precision    recall  f1-score   support

       False       1.00      0.98      0.99       989
        True       0.84      1.00      0.91       126

    accuracy                           0.98      1115
   macro avg       0.92      0.99      0.95      1115
weighted avg       0.98      0.98      0.98      1115



Random Forest

In [20]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier()
classifier.fit(X_train,y_train)
y_pred = classifier.predict(X_test)
score = classifier.score(X_test,y_test)
print(score)


0.979372197309417


In [21]:
print(classification_report(y_pred,y_test))

              precision    recall  f1-score   support

       False       1.00      0.98      0.99       988
        True       0.85      1.00      0.92       127

    accuracy                           0.98      1115
   macro avg       0.92      0.99      0.95      1115
weighted avg       0.98      0.98      0.98      1115



Word 2 Vec

In [23]:
#pretrained 300 dimension model
#import gensim.downloader as api

#wv = api.load('word2vec-google-news-300')

In [24]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [25]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [26]:
corpus = []

for i in range(0,len(messages)):
  review = re.sub('[^a-zA-Z]', '  ', messages['message'][i])
  review = review.lower()
  review = review.split()

  review = [lemmatizer.lemmatize(word) for word in review if not word in stopwords.words('english')]
  review = ' '.join(review)
  corpus.append(review)

In [27]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 31.8 MB/s  0:00:00


In [28]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [31]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [32]:
words = []
# for all the words in corpus
for sent in corpus :
  sent_token = sent_tokenize(sent) # tokenise each word
  for sent in sent_token : # for all the tokens -> preprocess it ( lower() )
    words.append(simple_preprocess(sent))

In [34]:
import gensim
model  = gensim.models.Word2Vec(words,
    window=10,
    min_count=2,
    vector_size=100,
)

In [35]:
model.wv.index_to_key

['call',
 'get',
 'ur',
 'gt',
 'lt',
 'go',
 'ok',
 'day',
 'free',
 'know',
 'come',
 'like',
 'time',
 'good',
 'got',
 'text',
 'love',
 'want',
 'send',
 'one',
 'need',
 'txt',
 'today',
 'going',
 'home',
 'stop',
 'lor',
 'sorry',
 'see',
 'mobile',
 'still',
 'take',
 'back',
 'da',
 'reply',
 'think',
 'tell',
 'dont',
 'week',
 'phone',
 'hi',
 'new',
 'later',
 'pls',
 'please',
 'co',
 'msg',
 'min',
 'dear',
 'make',
 'night',
 'message',
 'well',
 'say',
 'thing',
 'much',
 'claim',
 'great',
 'hope',
 'oh',
 'hey',
 'number',
 'happy',
 'friend',
 'wat',
 'give',
 'work',
 'way',
 'yes',
 'www',
 'let',
 'prize',
 'right',
 'tomorrow',
 'already',
 'tone',
 'ask',
 'said',
 'cash',
 'win',
 'yeah',
 'amp',
 'really',
 'life',
 'im',
 'babe',
 'meet',
 'find',
 'morning',
 'last',
 'service',
 'year',
 'uk',
 'thanks',
 'miss',
 'would',
 'com',
 'also',
 'nokia',
 'anything',
 'lol',
 'care',
 'every',
 'feel',
 'keep',
 'pick',
 'sure',
 'sent',
 'contact',
 'urgent',


In [36]:
model.corpus_count

5564

In [37]:
model.epochs

5

In [41]:
model.wv.similar_by_word('prize')

[('claim', 0.9995670318603516),
 ('guaranteed', 0.9993491768836975),
 ('land', 0.9991483688354492),
 ('urgent', 0.9990576505661011),
 ('line', 0.9990436434745789),
 ('draw', 0.9990413188934326),
 ('call', 0.9989827275276184),
 ('bonus', 0.9988529682159424),
 ('awarded', 0.9988387227058411),
 ('contact', 0.9988135099411011)]

In [42]:
model.wv['kid'].shape

(100,)

AVERAGE WORD TO VEC

In [70]:
doc_words = [simple_preprocess(doc) for doc in corpus]

model_aligned = gensim.models.Word2Vec(doc_words, window=10, min_count=2, vector_size=100)

# 3. Define a safe average Word2Vec function
def get_avg_vector(words_list, model):
    vectors = [model.wv[word] for word in words_list if word in model.wv.index_to_key]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

# 4. Generate features aligned with 'y'
X_aligned = np.array([get_avg_vector(w, model_aligned) for w in doc_words])
y_aligned = y  # Now both have length 5572

print(f"X shape: {X_aligned.shape}, y shape: {y_aligned.shape}")

X shape: (5572, 100), y shape: (5572,)


In [71]:
X_train, X_test, y_train, y_test = train_test_split(X_aligned, y_aligned, test_size=0.2, random_state=42)

# Since Word2Vec contains negative values, we use RandomForest instead of MultinomialNB
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
print("Accuracy:", rf_model.score(X_test, y_test))
print(classification_report(y_test, y_pred))

Accuracy: 0.9632286995515695
              precision    recall  f1-score   support

       False       0.97      0.99      0.98       965
        True       0.92      0.80      0.85       150

    accuracy                           0.96      1115
   macro avg       0.94      0.89      0.92      1115
weighted avg       0.96      0.96      0.96      1115

